
# SimulStreaming / AlignAtt STT server (Whisper small, Spanish)

Experimental **SimulStreaming** backend for the subtitle-overlay thesis. It replaces
**only** the STT engine in Colab: the board audio capture, WebSocket bridge, session
protocol, firmware ACKs and HDMI overlay are all reused unchanged.

**How to use:** `Runtime -> Run all`, wait until the last cell prints `HEALTH: ready`
and the public URLs, then from WSL run `./scripts/audiotestsimulstream.sh`.

- Upstream (pinned): https://github.com/ufal/SimulStreaming @ `077ea37d5ab4ff98bc567e4507f140dc4e5d5ad6`
- Engine: `simulstreaming_alignatt` (PyTorch Whisper + AlignAtt + VAC), **not** faster-whisper / CTranslate2
- Model: OpenAI Whisper multilingual **small** `.pt` from Drive, SHA-256 validated
- Do **not** mark ready until the model is loaded and a real warm-up decode finished.

## 1. Install dependencies

In [ ]:
# Colab already ships a CUDA torch. Install the server + SimulStreaming deps.
!pip -q install fastapi 'uvicorn[standard]' pyngrok websockets soundfile
UPSTREAM_COMMIT = "077ea37d5ab4ff98bc567e4507f140dc4e5d5ad6"
print("deps installed")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/TESIS/simulstreaming'
MODEL_PATH = f'{DRIVE_ROOT}/models/whisper-small.pt'
AUDIO_DIR  = f'{DRIVE_ROOT}/audio'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted. Expecting model at:', MODEL_PATH)

## 3. Clone this repo (branch `dev/simulstream`) + upstream SimulStreaming

The `dev/simulstream` branch must already be pushed to GitHub for Colab to clone
it. If this cell errors with "branch not found", push the branch first, then
re-run.

In [ ]:
import os, sys, subprocess
os.chdir('/content')
REPO_URL = 'https://github.com/Nacholazabal/subtitle_overlay_fw.git'
REPO_DIR = '/content/subtitle_overlay_fw'
BRANCH = 'dev/simulstream'

# Fail loudly and early if the branch is not on the remote (the usual cause of a
# later "No module named 'scripts'").
remote = subprocess.run(['git', 'ls-remote', '--heads', REPO_URL, BRANCH],
                        capture_output=True, text=True)
assert remote.stdout.strip(), (
    f"branch {BRANCH!r} not found on {REPO_URL} -- push it first:\n"
    f"    git push -u origin {BRANCH}"
)

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)
assert os.path.isdir(os.path.join(REPO_DIR, 'scripts')), 'clone did not produce scripts/'
print('repo at', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())

if not os.path.isdir('/content/SimulStreaming'):
    subprocess.run(['git', 'clone', 'https://github.com/ufal/SimulStreaming.git',
                    '/content/SimulStreaming'], check=True)
# Pin upstream to the exact commit this integration was written against.
subprocess.run(['git', '-C', '/content/SimulStreaming', 'checkout', '-q', "077ea37d5ab4ff98bc567e4507f140dc4e5d5ad6"], check=True)
subprocess.run(['git', '-C', '/content/SimulStreaming', '--no-pager', 'log', '-1', '--oneline'], check=True)
!pip -q install -r /content/SimulStreaming/requirements_whisper.txt || echo 'requirements_whisper install returned nonzero (often fine on Colab)'

for path in [REPO_DIR, '/content/SimulStreaming']:
    if path not in sys.path:
        sys.path.insert(0, path)
print('repos ready')

## 4. GPU + versions

In [ ]:
import torch, platform
print('python  :', platform.python_version())
print('torch   :', torch.__version__)
print('cuda    :', torch.version.cuda)
print('gpu     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU: set Runtime -> Change runtime type -> GPU'
upstream_sha = subprocess.check_output(['git','-C','/content/SimulStreaming','rev-parse','HEAD']).decode().strip()
assert upstream_sha == "077ea37d5ab4ff98bc567e4507f140dc4e5d5ad6", f'upstream not pinned: {upstream_sha}'
print('upstream:', upstream_sha)

## 5. Verify the checkpoint (exists + SHA-256)

Refuses a faster-whisper directory, a CTranslate2 model, `small.en`, or a corrupt file.
If the file is missing, run the **optional download** cell below once.

In [ ]:
import sys
for _p in ['/content/subtitle_overlay_fw', '/content/SimulStreaming']:
    if _p not in sys.path:
        sys.path.insert(0, _p)
from scripts.stt_simulstreaming_backend import MODEL_SHA256, validate_checkpoint, MODEL_DOWNLOAD_URL
import os
if not os.path.exists(MODEL_PATH):
    print('MISSING checkpoint at', MODEL_PATH)
    print('Run the optional download cell (cell 6) once to fetch it to Drive.')
else:
    sha = validate_checkpoint(MODEL_PATH, MODEL_SHA256)
    print('checkpoint OK, sha256=', sha)

## 6. (Optional) Download the official Whisper small checkpoint to Drive

Only needed the first time. It writes the file to Drive so later runs reuse it —
the notebook never downloads silently on every run.

In [ ]:
import sys
for _p in ['/content/subtitle_overlay_fw', '/content/SimulStreaming']:
    if _p not in sys.path:
        sys.path.insert(0, _p)
import os, urllib.request
from scripts.stt_simulstreaming_backend import MODEL_DOWNLOAD_URL, MODEL_SHA256, validate_checkpoint
if os.path.exists(MODEL_PATH):
    print('already present, skipping download')
else:
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    tmp = '/content/whisper-small.pt'
    print('downloading official OpenAI small.pt ...')
    urllib.request.urlretrieve(MODEL_DOWNLOAD_URL, tmp)
    validate_checkpoint(tmp, MODEL_SHA256)   # fail before copying a bad file to Drive
    import shutil; shutil.copy(tmp, MODEL_PATH)
    print('validated + copied to', MODEL_PATH)

## 7. Verify the three audios

In [ ]:
import os
EXPECTED = ['desay-short.webm', 'noticiero-short.webm', 'rel-short.webm']
missing = [n for n in EXPECTED if not os.path.exists(os.path.join(AUDIO_DIR, n))]
assert not missing, f'missing audios in {AUDIO_DIR}: {missing}'
print('all three audios present in', AUDIO_DIR)
# ffmpeg can convert on demand to WAV mono S16LE 16 kHz for a standalone check.

## 8. Load the model and run a REAL warm-up

Loading + warm-up happen here, synchronously. `/health` only becomes `ready` after
this succeeds. If loading fails, the full exception is shown and the server is NOT
started (no endless "connection refused").

In [ ]:
import sys
for _p in ['/content/subtitle_overlay_fw', '/content/SimulStreaming']:
    if _p not in sys.path:
        sys.path.insert(0, _p)
from scripts.stt_simulstreaming_server import ServerConfig, BackendState, create_app
from scripts.stt_simulstreaming_backend import SimulStreamingConfig

backend = SimulStreamingConfig(
    model='small', language='es', task='transcribe',
    min_chunk_sec=1.0, beams=1, use_vac=True,
    frame_threshold=25, audio_max_len=30.0, audio_min_len=0.0,
    never_fire=False, model_path=MODEL_PATH,
)
config = ServerConfig(host='0.0.0.0', port=8765, device='cuda', model_path=MODEL_PATH,
                      warmup_sec=1.0, backend=backend)

state = BackendState(config)
state.run_loader()   # synchronous: load + warm-up
import json
if not state.is_ready():
    print('BACKEND FAILED TO LOAD:\n')
    print(state.health_payload().get('error_detail', ''))
    raise RuntimeError('model load failed; fix the error above before serving')
print('backend ready. effective config:')
print(json.dumps(state.health_payload()['effective_config'], indent=2, ensure_ascii=False))

## 9. Start FastAPI/Uvicorn (background) and wait for /health ready

In [ ]:
import threading, time, uvicorn, requests
app = create_app(config, backend_state=state)  # startup won't reload (already ready)
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8765, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()

deadline = time.time() + 60
health = None
while time.time() < deadline:
    try:
        health = requests.get('http://127.0.0.1:8765/health', timeout=2).json()
        if health.get('ready'):
            break
    except Exception:
        pass
    time.sleep(0.5)
assert health and health.get('ready'), f'server did not become ready: {health}'
print('HEALTH: ready ->', health['run_engine'], health['model'], health['device'])

## 10. Open ngrok (only now that readiness is real) and print the URLs

In [ ]:
from pyngrok import ngrok, conf
import json
# Optional: set your authtoken / reserved static domain.
NGROK_AUTHTOKEN = ''   # <- paste if you use one
NGROK_DOMAIN = 'passage-capacity-wistful.ngrok-free.dev'  # the project's static tunnel; '' for a random URL
if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()
kwargs = {'proto': 'http', 'bind_tls': True}
if NGROK_DOMAIN:
    kwargs['domain'] = NGROK_DOMAIN
tunnel = ngrok.connect(8765, **kwargs)
http_url = tunnel.public_url
ws_url = http_url.replace('https://', 'wss://').replace('http://', 'ws://') + '/stt/stream'

print('='*70)
print('SimulStreaming STT server is LIVE')
print('  backend        :', health['run_engine'])
print('  model          :', health['model'], '(sha256', health['model_sha256'][:12] + '...)')
print('  device         :', health['device'])
print('  upstream commit:', health['upstream_commit'])
print('  effective cfg  :', json.dumps(health['effective_config']))
print('  HTTP  URL      :', http_url)
print('  HEALTH URL     :', http_url + '/health')
print('  WEBSOCKET URL  :', ws_url)
print('='*70)
print('From WSL:  STT_STREAM_URL="%s" ./scripts/audiotestsimulstream.sh' % ws_url)

## 11. Keep the server running

In [ ]:
import time
print('Server running. Leave this cell active. Interrupt to stop.')
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print('stopping')

## (Optional) Standalone offline test to Drive

Transcribes the three audios via `/stt/offline` and saves a small JSON to
`TESIS/simulstreaming/results/`. Marked `offline_proxy` — this is NOT a verified
human reference.

In [ ]:
import requests, os, json, time
out = {'run_engine': health['run_engine'], 'upstream_commit': health['upstream_commit'],
       'model_sha256': health['model_sha256'], 'reference_kind': 'offline_proxy', 'clips': {}}
for name in ['desay-short.webm', 'noticiero-short.webm', 'rel-short.webm']:
    with open(os.path.join(AUDIO_DIR, name), 'rb') as fh:
        data = fh.read()
    r = requests.post('http://127.0.0.1:8765/stt/offline', data=data,
                      headers={'X-Audio-Filename': name}, timeout=300)
    r.raise_for_status()
    out['clips'][name] = r.json()
    print(name, '->', out['clips'][name]['text'][:80])
path = os.path.join(RESULTS_DIR, f'offline_{int(time.time())}.json')
with open(path, 'w', encoding='utf-8') as fh:
    json.dump(out, fh, ensure_ascii=False, indent=2)
print('saved', path)